In [ ]:
"""
Plik: train_yolo_colab.ipynb (Komórka wykonawcza Google Colab)
Opis: Skrypt dedykowany do uruchomienia w chmurowym środowisku obliczeniowym (Google Colab).
Realizuje zautomatyzowany potok (pipeline) przygotowania danych i uczenia modelu YOLOv8:
1. Montowanie przestrzeni dyskowej Google Drive.
2. Szybka ekstrakcja zestawu danych z archiwum ZIP na lokalny dysk maszyny wirtualnej.
3. Dynamiczna generacja pliku konfiguracyjnego data.yaml w standardzie POSIX (omijająca
   konflikty ścieżek między systemami Windows a Linux).
4. Inicjacja procesu uczenia sieci na akceleratorze GPU z ciągłą synchronizacją 
   wyników z powrotem do chmury.
"""
# 1. Montowanie dysku
from google.colab import drive
drive.mount('/content/drive')

# 2. Instalacja biblioteki
!pip install ultralytics -q
from ultralytics import YOLO
import shutil

# 3. Kopiowanie i rozpakowywanie danych z Google Drive
zip_path = "/content/drive/MyDrive/CvFootballTracker_Data/Detection/yoloformat_large.zip"
local_data_dir = "/content/yoloformat"

print("Rozpakowywanie archiwum...")
shutil.unpack_archive(zip_path, extract_dir="/content/")
print("Dane rozpakowane!")

# 4. Nadpisanie pliku data.yaml (Rozwiązuje problem windowsowych ścieżek z Twojego komputera)
# Używamy bezpośredniego wskazania na foldery (train, valid, test) zamiast plików .txt
yaml_content = f"""
path: {local_data_dir}
train: train
val: valid
test: test

names:
  0: ball
  1: team_1
  2: team_2
  3: referee
"""

with open(f"{local_data_dir}/data.yaml", 'w', encoding='utf-8') as f:
    f.write(yaml_content.strip())
print("Plik data.yaml został nadpisany ścieżkami Linux (Colab).")

# 5. Odpalenie treningu
print("🔥 Rozpoczynamy trening YOLOv8 na GPU...")
model = YOLO("yolov8n.pt")

results = model.train(
    data=f"{local_data_dir}/data.yaml",
    epochs=100,         # Możesz zwiększyć docelowo np. do 50 lub 100
    imgsz=640,
    batch=16,          
    project="/content/drive/MyDrive/CvFootballTracker_Data/results", # Zapis do chmury
    name="yolo_colab_run_det_100e_L"
)